# Modular Financial Agent Walkthrough

## Advanced Multi-Tool Coordination Using Helper Modules Architecture

This notebook provides a comprehensive walkthrough of the **Modular Financial Agent** system, demonstrating how the helper modules work together to create a sophisticated financial analysis platform. The system features:

- **🏗️ Modular Architecture**: Clean separation using helper modules
- **📄 Document Analysis**: SEC 10-K filing analysis via DocumentToolsManager
- **🔧 Function Tools**: Database queries, market data, and PII protection via FunctionToolsManager  
- **🧠 Intelligent Coordination**: AgentCoordinator with smart routing and automatic PII protection
- **🔄 Seamless Integration**: All components work together automatically

### 🏛️ Architecture Overview

The system uses a **3-module architecture**:

#### **1. DocumentToolsManager** (`helper_modules/document_tools.py`)
- Creates 3 QueryEngineTool objects for AAPL, GOOGL, TSLA 10-K filings
- Each tool analyzes specific company SEC documents
- Powered by LlamaIndex with OpenAI embeddings

#### **2. FunctionToolsManager** (`helper_modules/function_tools.py`)  
- Creates 3 function tools for different capabilities:
  - `database_query_tool` - SQL generation and execution with column output
  - `finance_market_search_tool` - Real-time Yahoo Finance data
  - `pii_protection_tool` - Automatic sensitive data masking

#### **3. AgentCoordinator** (`helper_modules/agent_coordinator.py`)
- Orchestrates all tools with intelligent routing
- Automatically detects when PII protection is needed
- Synthesizes results from multiple tools
- Provides clean backward compatibility

Let's explore how this modular system delivers comprehensive financial insights!

## 1. Environment Setup and Module Imports

First, let's set up our environment and import the modular components. The beauty of this architecture is that each module can be used independently or together.

In [1]:
# Import required libraries
import os
import sys
from pathlib import Path
import pandas as pd

# Add current directory to Python path for imports
current_dir = Path().absolute()
if str(current_dir) not in sys.path:
    sys.path.append(str(current_dir))

# Import the modular helper modules
from helper_modules.document_tools import DocumentToolsManager
from helper_modules.function_tools import FunctionToolsManager  
from helper_modules.agent_coordinator import AgentCoordinator

print("✅ All helper modules imported successfully!")
print(f"📁 Working directory: {current_dir}")

# Verify required files exist
required_files = [
    "data/financial.db",
    "data/10k_documents/AAPL_10K_2024.pdf",
    "data/10k_documents/GOOGL_10K_2024.pdf", 
    "data/10k_documents/TSLA_10K_2024.pdf"
]

print("\n🔍 Checking required files:")
all_files_exist = True
for file_path in required_files:
    exists = Path(file_path).exists()
    status = "✅" if exists else "❌"
    print(f"   {status} {file_path}")
    if not exists:
        all_files_exist = False

print(f"\n💡 OpenAI API Key configured: {'✅' if os.getenv('OPENAI_API_KEY') else '❌'}")
print(f"📋 System ready: {'✅' if all_files_exist else '❌'}")

✅ All helper modules imported successfully!
📁 Working directory: /Users/cxnbf/sbk/udacity-agentic-ai-for-finance-course-projects/course-3/starter-code/starter_code

🔍 Checking required files:
   ✅ data/financial.db
   ✅ data/10k_documents/AAPL_10K_2024.pdf
   ✅ data/10k_documents/GOOGL_10K_2024.pdf
   ✅ data/10k_documents/TSLA_10K_2024.pdf

💡 OpenAI API Key configured: ✅
📋 System ready: ✅


## 2. Module 1: DocumentToolsManager - PDF Document Analysis

Let's start by exploring the `DocumentToolsManager`. This module creates specialized tools for analyzing SEC 10-K filings for each company.

In [2]:
# Initialize the DocumentToolsManager
print("📄 Initializing DocumentToolsManager...")
print("=" * 50)

doc_manager = DocumentToolsManager()
print(f"✅ DocumentToolsManager created")

# Build the document tools
print("\n🔧 Building document analysis tools...")
document_tools = doc_manager.build_document_tools()

print(f"\n📊 Document Tools Created:")
for i, tool in enumerate(document_tools, 1):
    print(f"   {i}. {tool.metadata.name}")
    print(f"      Description: {tool.metadata.description}")
    print(f"      Type: {type(tool).__name__}")
    print()

print(f"🎯 Total Document Tools: {len(document_tools)}")
print("=" * 50)

📄 Initializing DocumentToolsManager...


✅ DocumentToolsManager created

🔧 Building document analysis tools...

📊 Document Tools Created:
   1. AAPL_10k_filing_tool
      Description: Answers questions about Apple Inc. (AAPL) from its 2024 SEC 10-K filing: business segments, products, revenue drivers, risk factors, competition and financial disclosures.
      Type: QueryEngineTool

   2. GOOGL_10k_filing_tool
      Description: Answers questions about Alphabet Inc. (Google) (GOOGL) from its 2024 SEC 10-K filing: business segments, products, revenue drivers, risk factors, competition and financial disclosures.
      Type: QueryEngineTool

   3. TSLA_10k_filing_tool
      Description: Answers questions about Tesla Inc. (TSLA) from its 2024 SEC 10-K filing: business segments, products, revenue drivers, risk factors, competition and financial disclosures.
      Type: QueryEngineTool

🎯 Total Document Tools: 3


In [3]:
# Test individual document tools
print("🧪 Testing Document Tools Individually")
print("=" * 50)

# Test Apple 10-K analysis
print("🍎 Testing Apple 10-K Tool:")
apple_tool = document_tools[0]  # First tool should be Apple
apple_query = "What are Apple's main revenue sources according to their 10-K filing?"

print(f"Query: {apple_query}")
try:
    apple_result = apple_tool.query_engine.query(apple_query)
    print(f"Result: {str(apple_result)[:300]}...")
    print("✅ Apple document tool working!")
except Exception as e:
    print(f"❌ Error: {e}")

print("\n" + "-" * 40)

# Test Google 10-K analysis  
print("🌐 Testing Google 10-K Tool:")
google_tool = document_tools[1]  # Second tool should be Google
google_query = "What business segments does Google operate in?"

print(f"Query: {google_query}")
try:
    google_result = google_tool.query_engine.query(google_query)
    print(f"Result: {str(google_result)[:300]}...")
    print("✅ Google document tool working!")
except Exception as e:
    print(f"❌ Error: {e}")

print("=" * 40)

# Test Tesla 10-K analysis
print("🚗 Testing Tesla 10-K Tool:")
tesla_tool = document_tools[2]  # Third tool should be Tesla
tesla_query = "What are Tesla's main revenue sources according to their 10-K filing?"

print(f"Query: {tesla_query}")
try:
    tesla_result = tesla_tool.query_engine.query(tesla_query)
    print(f"Result: {str(tesla_result)[:300]}...")
    print("✅ Tesla document tool working!")
except Exception as e:
    print(f"❌ Error: {e}")

print("=" * 50)

🧪 Testing Document Tools Individually
🍎 Testing Apple 10-K Tool:
Query: What are Apple's main revenue sources according to their 10-K filing?


Result: - Products, including smartphones, personal computers, tablets, wearables, and accessories
- Services, including advertising, AppleCare, cloud services, digital content, and payment services
- Net sales from products totaled $294,866 million in 2024
- Net sales from services totaled $96,169 million ...
✅ Apple document tool working!

----------------------------------------
🌐 Testing Google 10-K Tool:
Query: What business segments does Google operate in?


Result: - Google Services
- Google Cloud
- Other Bets...
✅ Google document tool working!
🚗 Testing Tesla 10-K Tool:
Query: What are Tesla's main revenue sources according to their 10-K filing?


Result: - Automotive sales
- Automotive regulatory credits
- Energy generation and storage
- Services and other...
✅ Tesla document tool working!


## 3. Module 2: FunctionToolsManager - Dynamic Function Tools

Now let's explore the `FunctionToolsManager`. This module creates three powerful function tools for database queries, market data, and PII protection.

In [4]:
# Initialize the FunctionToolsManager
print("🔧 Initializing FunctionToolsManager...")
print("=" * 50)

func_manager = FunctionToolsManager()
print(f"✅ FunctionToolsManager created")
print(f"💾 Database path: {func_manager.db_path}")

# Create the function tools
print("\n🛠️ Creating function tools...")
function_tools = func_manager.create_function_tools()

print(f"\n📊 Function Tools Created:")
for i, tool in enumerate(function_tools, 1):
    print(f"   {i}. {tool.metadata.name}")
    print(f"      Description: {tool.metadata.description}")
    print(f"      Function: {tool.fn.__name__}")
    print()

print(f"🎯 Total Function Tools: {len(function_tools)}")
print("=" * 50)

🔧 Initializing FunctionToolsManager...
✅ FunctionToolsManager created
💾 Database path: /Users/cxnbf/sbk/udacity-agentic-ai-for-finance-course-projects/course-3/starter-code/starter_code/data/financial.db

🛠️ Creating function tools...

📊 Function Tools Created:
   1. database_query_tool
      Description: Query the customer and portfolio database in natural language: customers, holdings, account balances, investment profiles, company records.
      Function: database_query_tool

   2. finance_market_search_tool
      Description: Real-time stock price, change and trading volume for Apple (AAPL), Google (GOOGL) and Tesla (TSLA) from Yahoo Finance.
      Function: finance_market_search_tool

   3. pii_protection_tool
      Description: Mask personal data (names, emails, phones, addresses, SSNs) in database results.
      Function: pii_protection_tool

   4. financial_analysis_tool
      Description: Technical analysis of AAPL, GOOGL or TSLA price history: period return, annualized volati

In [5]:
# Test individual function tools
print("🧪 Testing Function Tools Individually")
print("=" * 50)

# Test Database Query Tool
print("💾 Testing Database Query Tool:")
db_tool = next(tool for tool in function_tools if "database" in tool.metadata.name)
db_query = "How many customers are in the database?"

print(f"Query: {db_query}")
try:
    db_result = db_tool.fn(db_query)
    print(f"Result: {db_result}")
    print("✅ Database tool working!")
except Exception as e:
    print(f"❌ Error: {e}")

print("\n" + "-" * 40)

# Test Market Data Tool
print("📈 Testing Market Data Tool:")
market_tool = next(tool for tool in function_tools if "market" in tool.metadata.name)
market_query = "What is Apple's current stock price?"

print(f"Query: {market_query}")
try:
    market_result = market_tool.fn(market_query)
    print(f"Result: {market_result}")
    print("✅ Market data tool working!")
except Exception as e:
    print(f"❌ Error: {e}")

print("\n" + "-" * 40)

# Test PII Protection Tool
print("🔒 Testing PII Protection Tool:")
pii_tool = next(tool for tool in function_tools if "pii" in tool.metadata.name)
test_data = "Customer: John Doe, Email: john.doe@example.com, Phone: 555-1234"
test_columns = "['customer_name', 'email', 'phone']"

print(f"Test Data: {test_data}")
print(f"Columns: {test_columns}")
try:
    pii_result = pii_tool.fn(test_data, test_columns)
    print(f"Protected Result: {pii_result}")
    print("✅ PII protection tool working!")
except Exception as e:
    print(f"❌ Error: {e}")

print("=" * 50)

🧪 Testing Function Tools Individually
💾 Testing Database Query Tool:
Query: How many customers are in the database?


Result: SQL Query: SELECT COUNT(id) FROM customers

Database Results:
{'COUNT(id)': 10}
COLUMNS: ['COUNT(id)']
✅ Database tool working!

----------------------------------------
📈 Testing Market Data Tool:
Query: What is Apple's current stock price?


Result: Real-Time Market Data:
AAPL: $336.67 | Change: +3.04 (+0.91%) | Volume: 33,380,854 [source: Yahoo Finance]
✅ Market data tool working!

----------------------------------------
🔒 Testing PII Protection Tool:
Test Data: Customer: John Doe, Email: john.doe@example.com, Phone: 555-1234
Columns: ['customer_name', 'email', 'phone']
Protected Result: Customer: John Doe, Email: ***@example.com, Phone: ***-***-1234

🔒 PII Protection Applied — masked fields: ['pattern-matched contact data']
✅ PII protection tool working!


## 4. Module 3: AgentCoordinator - Intelligent Orchestration

Now let's see the real power - the `AgentCoordinator` that intelligently orchestrates all tools together, providing automatic routing and PII protection coordination.

In [6]:
# Initialize the AgentCoordinator
print("🧠 Initializing AgentCoordinator...")
print("=" * 50)

# Create coordinator with verbose output to see the process
coordinator = AgentCoordinator(verbose=True)

# Setup the coordinator - it automatically creates all tools via both managers
coordinator.setup()

print(f"\n📊 Tools available: {coordinator.list_available_tools()}")
print("=" * 50)

🧠 Initializing AgentCoordinator...
   ✅ AAPL_10k_filing_tool ready
   ✅ GOOGL_10k_filing_tool ready
   ✅ TSLA_10k_filing_tool ready
✅ Ready: 3 document tools, 5 function tools

📊 Tools available: ['AAPL_10k_filing_tool', 'GOOGL_10k_filing_tool', 'TSLA_10k_filing_tool', 'database_query_tool', 'finance_market_search_tool', 'pii_protection_tool', 'financial_analysis_tool', 'portfolio_analysis_tool']


In [7]:
# Explore the created tools within the coordinator
print("🔍 Exploring Coordinator's Tool Ecosystem")
print("=" * 50)

print("📄 Document Tools in Coordinator:")
for i, tool in enumerate(coordinator.document_tools, 1):
    print(f"   {i}. {tool.metadata.name}: {tool.metadata.description}")

print("\n🔧 Function Tools in Coordinator:")
for i, tool in enumerate(coordinator.function_tools, 1):
    print(f"   {i}. {tool.metadata.name}: {tool.metadata.description}")

print(f"\n📋 Total Tools: {len(coordinator.list_available_tools())}")
print("=" * 50)

🔍 Exploring Coordinator's Tool Ecosystem
📄 Document Tools in Coordinator:
   1. AAPL_10k_filing_tool: Answers questions about Apple Inc. (AAPL) from its 2024 SEC 10-K filing: business segments, products, revenue drivers, risk factors, competition and financial disclosures.
   2. GOOGL_10k_filing_tool: Answers questions about Alphabet Inc. (Google) (GOOGL) from its 2024 SEC 10-K filing: business segments, products, revenue drivers, risk factors, competition and financial disclosures.
   3. TSLA_10k_filing_tool: Answers questions about Tesla Inc. (TSLA) from its 2024 SEC 10-K filing: business segments, products, revenue drivers, risk factors, competition and financial disclosures.

🔧 Function Tools in Coordinator:
   1. database_query_tool: Query the customer and portfolio database in natural language: customers, holdings, account balances, investment profiles, company records.
   2. finance_market_search_tool: Real-time stock price, change and trading volume for Apple (AAPL), Google (GO

## 5. Single Tool Routing - Document Analysis

Let's test how the coordinator intelligently routes queries to individual tools. We'll start with document-based queries.

In [8]:
# Test document tool routing
print("📄 Testing Document Tool Routing")
print("=" * 50)

# Apple-specific query
print("🍎 Apple 10-K Analysis:")
apple_query = "What are Apple's main business segments and how do they generate revenue?"
print(f"Query: {apple_query}")

apple_response = coordinator.query(apple_query, verbose=True)
print(f"\nFinal Response: {apple_response}")
print("\n" + "=" * 50)

# Google-specific query
print("🌐 Google 10-K Analysis:")
google_query = "What are the key risk factors mentioned in Google's 10-K filing?"
print(f"Query: {google_query}")

google_response = coordinator.query(google_query, verbose=True)
print(f"\nFinal Response: {google_response}")
print("\n" + "=" * 50)

📄 Testing Document Tool Routing
🍎 Apple 10-K Analysis:
Query: What are Apple's main business segments and how do they generate revenue?


🧭 Routed to ['AAPL_10k_filing_tool'] (confidence 1.00)



Final Response: - Apple's main business segments are the Americas, Europe, Greater China, Japan, and Rest of Asia Pacific.
- Revenue is generated through direct and indirect distribution channels, with 38% of net sales coming from direct channels and 62% from indirect channels.
- The Americas segment includes North and South America, while Europe includes European countries, India, the Middle East, and Africa.
- Greater China includes China mainland, Hong Kong, and Taiwan, with iPhone revenue representing a moderately higher proportion of net sales in this segment.
- Rest of Asia Pacific includes Australia and other Asian countries not included in the other reportable segments.

🌐 Google 10-K Analysis:
Query: What are the key risk factors mentioned in Google's 10-K filing?


🧭 Routed to ['GOOGL_10k_filing_tool'] (confidence 1.00)



Final Response: - Intellectual property rights protection is crucial for Google, as any inability to protect them could reduce the value of their products, services, and brands.
- Maintaining and enhancing Google's strong brands is essential for expanding their base of users, advertisers, customers, content providers, and other partners.
- Google faces manufacturing and supply chain risks that could harm their business, financial condition, and operating results.
- The path to achieving net-zero emissions and 24/7 carbon-free energy by 2030 is challenging and may not be easy or linear.



## 6. Single Tool Routing - Function Tools

Now let's test how the coordinator routes to function tools for database queries and market data.

In [9]:
# Test database tool routing
print("💾 Testing Database Tool Routing")
print("=" * 50)

# Database query without PII
print("📊 Non-PII Database Query:")
db_query = "How many customers are in the database grouped by investment profile?"
print(f"Query: {db_query}")

db_response = coordinator.query(db_query, verbose=True)
print(f"\nFinal Response: {db_response}")
print("\n" + "=" * 50)

# Market data query
print("📈 Testing Market Data Tool Routing:")
market_query = "What is Tesla's current stock price and trading volume?"
print(f"Query: {market_query}")

market_response = coordinator.query(market_query, verbose=True)
print(f"\nFinal Response: {market_response}")
print("=" * 50)

💾 Testing Database Tool Routing
📊 Non-PII Database Query:
Query: How many customers are in the database grouped by investment profile?


🧭 Routed to ['database_query_tool'] (confidence 1.00)



Final Response: SQL Query: SELECT investment_profile, COUNT(DISTINCT customers.id) AS customer_count
FROM customers
GROUP BY investment_profile

Database Results:
{'investment_profile': 'aggressive', 'customer_count': 3}
{'investment_profile': 'conservative', 'customer_count': 3}
{'investment_profile': 'moderate', 'customer_count': 4}
COLUMNS: ['investment_profile', 'customer_count']

📈 Testing Market Data Tool Routing:
Query: What is Tesla's current stock price and trading volume?


🧭 Routed to ['finance_market_search_tool'] (confidence 1.00)



Final Response: Real-Time Market Data:
TSLA: $377.81 | Change: -2.87 (-0.75%) | Volume: 25,496,215 [source: Yahoo Finance]


## 7. Automatic PII Protection Coordination

One of the most sophisticated features is the automatic PII protection. Let's test how the coordinator automatically detects when database results contain PII and applies protection.

In [10]:
# Test automatic PII protection coordination
print("🔒 Testing Automatic PII Protection Coordination")
print("=" * 60)

# Query that will return PII data
print("🧪 Test: Query WITH PII Fields (should trigger automatic protection)")
print("-" * 50)

pii_query = "Show me customer details including names and emails for customers who own Apple stock"
print(f"Query: {pii_query}")

pii_response = coordinator.query(pii_query, verbose=True)
print(f"\nFinal Response: {pii_response}")

print("\n" + "=" * 60)

# Query that doesn't return PII data
print("🧪 Test: Query WITHOUT PII Fields (should NOT trigger protection)")
print("-" * 50)

no_pii_query = "Show portfolio values and investment profiles without customer details"
print(f"Query: {no_pii_query}")

no_pii_response = coordinator.query(no_pii_query, verbose=True)
print(f"\nFinal Response: {no_pii_response}")

print("\n🎯 Key Observations:")
print("✅ PII protection automatically applied when needed")
print("✅ No protection overhead for non-PII queries")
print("✅ Clean separation between data retrieval and privacy protection")
print("=" * 60)

🔒 Testing Automatic PII Protection Coordination
🧪 Test: Query WITH PII Fields (should trigger automatic protection)
--------------------------------------------------
Query: Show me customer details including names and emails for customers who own Apple stock


🧭 Routed to ['database_query_tool'] (confidence 1.00)


   🔒 PII detected ['email', 'first_name', 'last_name'] — applying protection

Final Response: SQL Query: SELECT customers.id, customers.first_name, customers.last_name, customers.email
FROM customers
JOIN portfolio_holdings ON customers.id = portfolio_holdings.customer_id
JOIN companies ON portfolio_holdings.symbol = companies.symbol
WHERE companies.symbol = 'AAPL'

Database Results:
{'id': 1, 'first_name': '****', 'last_name': '****', 'email': '***@email.com'}
{'id': 2, 'first_name': '****', 'last_name': '****', 'email': '***@email.com'}
{'id': 3, 'first_name': '****', 'last_name': '****', 'email': '***@email.com'}
{'id': 4, 'first_name': '****', 'last_name': '****', 'email': '***@email.com'}
{'id': 5, 'first_name': '****', 'last_name': '****', 'email': '***@email.com'}
{'id': 6, 'first_name': '****', 'last_name': '****', 'email': '***@email.com'}
{'id': 7, 'first_name': '****', 'last_name': '****', 'email': '***@email.com'}
{'id': 8, 'first_name': '****', 'last_name': '****', 'email'

🧭 Routed to ['portfolio_analysis_tool'] (confidence 1.00)



Final Response: Portfolio Analysis:

Customer 1 (moderate, medium risk):
  AAPL: 50.00 sh @ $150.00/sh paid, now $336.67/sh | Total cost $7,500.00 | Value $16,833.50 (Yahoo Finance) | P&L +9,333.50 (+124.45%) | Weight 70.6%
  GOOGL: 20.00 sh @ $2,500.00/sh paid, now $350.50/sh | Total cost $50,000.00 | Value $7,010.00 (Yahoo Finance) | P&L -42,990.00 (-85.98%) | Weight 29.4%
  TOTAL: Cost $57,500.00 | Value $23,843.50 | P&L -33,656.50 (-58.53%)
  ALERT: AAPL 71% of portfolio — over-concentrated for medium risk tolerance (max 60%)

Customer 2 (aggressive, high risk):
  AAPL: 100.00 sh @ $145.00/sh paid, now $336.67/sh | Total cost $14,500.00 | Value $33,667.00 (Yahoo Finance) | P&L +19,167.00 (+132.19%) | Weight 44.3%
  TSLA: 75.00 sh @ $800.00/sh paid, now $377.81/sh | Total cost $60,000.00 | Value $28,335.75 (Yahoo Finance) | P&L -31,664.25 (-52.77%) | Weight 37.3%
  GOOGL: 40.00 sh @ $2,400.00/sh paid, now $350.50/sh | Total cost $96,000.00 | Value $14,020.00 (Yahoo Finance) | P&L -

## 8. Multi-Tool Coordination - The Real Power

Now let's see the coordinator's most impressive capability: intelligently combining multiple tools to answer complex queries that require different data sources.

In [11]:
# Test dual tool coordination: Database + Market Data
print("🔄 Testing Dual Tool Coordination: Database + Market Data")
print("=" * 60)

dual_query = "Compare the current Tesla stock price with what our Tesla-holding customers paid based on portfolio data"
print(f"Query: {dual_query}")

dual_response = coordinator.query(dual_query, verbose=True)
print(f"\nFinal Response: {dual_response}")
print("=" * 60)

🔄 Testing Dual Tool Coordination: Database + Market Data
Query: Compare the current Tesla stock price with what our Tesla-holding customers paid based on portfolio data


🧭 Routed to ['portfolio_analysis_tool', 'finance_market_search_tool'] (confidence 1.00)



Final Response: 1. The current Tesla stock price is $377.81, while our Tesla-holding customers paid prices ranging from $750.00 to $860.00 per share.

---

**portfolio_analysis_tool:**
- Customer 2 paid $800.00 per share for 75.00 shares of TSLA.
- Customer 5 paid $750.00 per share for 150.00 shares of TSLA.
- Customer 8 paid $780.00 per share for 100.00 shares of TSLA.
- Customer 10 paid $860.00 per share for 25.00 shares of TSLA.

**finance_market_search_tool:**
- Current Tesla stock price: $377.81.


In [12]:
# Test triple tool coordination: Document + Database + Market
print("🌟 Testing Triple Tool Coordination: Document + Database + Market")
print("=" * 60)

triple_query = "Analyze Apple comprehensively: show me Apple's business segments from their 10-K, current stock performance, and which of our customers own Apple stock"
print(f"Query: {triple_query}")

triple_response = coordinator.query(triple_query, verbose=True)
print(f"\nFinal Response: {triple_response}")
print("=" * 60)

🌟 Testing Triple Tool Coordination: Document + Database + Market
Query: Analyze Apple comprehensively: show me Apple's business segments from their 10-K, current stock performance, and which of our customers own Apple stock


🧭 Routed to ['AAPL_10k_filing_tool', 'finance_market_search_tool', 'database_query_tool'] (confidence 1.00)


   🔒 PII detected ['company_name', 'email', 'first_name', 'last_name', 'phone'] — applying protection



Final Response: Apple's business segments from their 10-K filing are the Americas, Europe, Greater China, Japan, and Rest of Asia Pacific. 

**AAPL_10k_filing_tool:**
- Apple's stock performance has outperformed both the S&P 500 Index and the Dow Jones U.S. Technology Supersector Index over the past five years.
- Apple's customers across various markets, including consumer, small and mid-sized business, education, enterprise, and government, may own Apple stock based on their interest in the company's products and services.

**finance_market_search_tool:**
- Apple's current stock price is $336.67 with a change of +3.04 (+0.91%) and a trading volume of 33,380,854 shares.

**database_query_tool:**
- A customer with a moderate investment profile and medium risk tolerance owns 50 shares of Apple (AAPL) purchased at $150 per share, with a current value of $8,750. 
- The customer's total portfolio value is $75,000, and the current stock price of Apple is $167.14.

No data gaps or conflicts 

## 9. Advanced Database Queries with Automatic SQL Generation

Let's explore the sophisticated database capabilities including automatic SQL generation and smart result handling.

In [13]:
# Test complex database queries
print("🎯 Testing Advanced Database Capabilities")
print("=" * 50)

# Complex aggregation query
print("📊 Complex Aggregation Query:")
complex_query = "What's the average portfolio value for each investment profile and how many customers are in each category?"
print(f"Query: {complex_query}")

complex_response = coordinator.query(complex_query, verbose=True)
print(f"\nResponse: {complex_response}")

print("\n" + "-" * 40)

# Multi-table join query
print("🔗 Multi-table Join Query:")
join_query = "Show the top 5 customers by total portfolio value along with their stock holdings details"
print(f"Query: {join_query}")

join_response = coordinator.query(join_query, verbose=True)
print(f"\nResponse: {join_response}")

print("=" * 50)

🎯 Testing Advanced Database Capabilities
📊 Complex Aggregation Query:
Query: What's the average portfolio value for each investment profile and how many customers are in each category?


🧭 Routed to ['database_query_tool'] (confidence 1.00)



Response: SQL Query: SELECT customers.investment_profile, COUNT(customers.id) AS customer_count, AVG(customers.total_portfolio_value) AS avg_portfolio_value
FROM customers
GROUP BY customers.investment_profile

Database Results:
{'investment_profile': 'aggressive', 'customer_count': 3, 'avg_portfolio_value': 193333.33333333334}
{'investment_profile': 'conservative', 'customer_count': 3, 'avg_portfolio_value': 38333.333333333336}
{'investment_profile': 'moderate', 'customer_count': 4, 'avg_portfolio_value': 82500.0}
COLUMNS: ['investment_profile', 'customer_count', 'avg_portfolio_value']

----------------------------------------
🔗 Multi-table Join Query:
Query: Show the top 5 customers by total portfolio value along with their stock holdings details


🧭 Routed to ['database_query_tool'] (confidence 1.00)


   🔒 PII detected ['first_name', 'last_name'] — applying protection

Response: SQL Query: SELECT c.id, c.first_name, c.last_name, ph.symbol, ph.shares, ph.purchase_price, md.close_price
FROM customers c
JOIN portfolio_holdings ph ON c.id = ph.customer_id
JOIN companies com ON ph.symbol = com.symbol
JOIN market_data md ON com.symbol = md.symbol
WHERE md.date = (SELECT MAX(date) FROM market_data WHERE symbol = com.symbol)
ORDER BY (ph.shares * ph.purchase_price * md.close_price) DESC
LIMIT 5

Database Results:
{'id': 5, 'first_name': '****', 'last_name': '****', 'symbol': 'GOOGL', 'shares': 80.0, 'purchase_price': 2300.0, 'close_price': 167.32242126718992}
{'id': 5, 'first_name': '****', 'last_name': '****', 'symbol': 'TSLA', 'shares': 150.0, 'purchase_price': 750.0, 'close_price': 228.91767100595015}
{'id': 8, 'first_name': '****', 'last_name': '****', 'symbol': 'GOOGL', 'shares': 60.0, 'purchase_price': 2450.0, 'close_price': 167.32242126718992}
{'id': 8, 'first_name': '****', 'last_na

## 10. System Architecture Analysis

Let's analyze what makes this modular architecture so powerful and examine the internal coordination mechanisms.

In [14]:
# Analyze the system architecture
print("🏗️ System Architecture Analysis")
print("=" * 50)

print("📊 Module Breakdown:")
print(f"   DocumentToolsManager: {len(coordinator.document_tools)} tools")
print(f"   FunctionToolsManager: {len(coordinator.function_tools)} tools")
print(f"   AgentCoordinator: 1 orchestrator")

print(f"\n🧠 Coordinator Intelligence Features:")
print("   ✅ Automatic tool selection via LLM routing")
print("   ✅ Multi-tool result synthesis")
print("   ✅ Automatic PII detection and protection")
print("   ✅ Error handling and graceful fallbacks")
print("   ✅ Clean modular architecture")

print(f"\n🔄 Tool Coordination Flow:")
print("   1. Query received by AgentCoordinator")
print("   2. LLM analyzes query and selects appropriate tools")
print("   3. Selected tools execute in sequence")
print("   4. Database results checked for PII fields")
print("   5. PII protection applied automatically if needed")
print("   6. Results synthesized into final response")

print(f"\n🎯 Key Architecture Benefits:")
print("   ✅ Clean separation of concerns")
print("   ✅ Each module can be used independently")
print("   ✅ Easy to test, maintain, and extend")
print("   ✅ Automatic coordination reduces complexity")
print("   ✅ Backward compatibility with existing code")

print("=" * 50)

🏗️ System Architecture Analysis
📊 Module Breakdown:
   DocumentToolsManager: 3 tools
   FunctionToolsManager: 5 tools
   AgentCoordinator: 1 orchestrator

🧠 Coordinator Intelligence Features:
   ✅ Automatic tool selection via LLM routing
   ✅ Multi-tool result synthesis
   ✅ Automatic PII detection and protection
   ✅ Error handling and graceful fallbacks
   ✅ Clean modular architecture

🔄 Tool Coordination Flow:
   1. Query received by AgentCoordinator
   2. LLM analyzes query and selects appropriate tools
   3. Selected tools execute in sequence
   4. Database results checked for PII fields
   5. PII protection applied automatically if needed
   6. Results synthesized into final response

🎯 Key Architecture Benefits:
   ✅ Clean separation of concerns
   ✅ Each module can be used independently
   ✅ Easy to test, maintain, and extend
   ✅ Automatic coordination reduces complexity
   ✅ Backward compatibility with existing code


## 11. Extras

The following cells show the additional capabilities built into this agent:
configuration management, financial and portfolio analytics, confidence-scored routing,
observability metrics, and a DSPy-powered feedback loop for improving the router.

### (a) Configuration Management

`config.toml` centralises models, providers, pricing, chunking, routing guidelines and tool descriptions.
The `active_provider()` function probes each provider in `provider_order` and returns the first that responds.

In [15]:
from helper_modules.models import active_provider, CONFIG

p = active_provider()
print("Active provider:", p.name)
print("  API base  :", p.api_base)
print("  LLM model :", p.llm)
print("  Embedding :", p.embedding)

print("\nProvider order :", CONFIG["models"]["provider_order"])
print("Temperature    :", CONFIG["models"]["temperature"])
print("Chunk size     :", CONFIG["documents"]["chunk_size"])
print("Chunk overlap  :", CONFIG["documents"]["chunk_overlap"])
print("Similarity top-k:", CONFIG["documents"]["similarity_top_k"])
print("Low-conf thresh:", CONFIG["routing"]["low_confidence_threshold"])
print("Pricing ($/1M)  LLM-in={llm_input}  LLM-out={llm_output}  embed={embedding}".format(
    **CONFIG["pricing"]))

Active provider: vocareum
  API base  : https://openai.vocareum.com/v1
  LLM model : gpt-3.5-turbo
  Embedding : text-embedding-ada-002

Provider order : ['vocareum', 'openrouter']
Temperature    : 0
Chunk size     : 1024
Chunk overlap  : 100
Similarity top-k: 3
Low-conf thresh: 0.6
Pricing ($/1M)  LLM-in=0.5  LLM-out=1.5  embed=0.1


### (b) Financial Analysis Tool

`financial_analysis_tool` calculates period return, annualised volatility, 20- and 50-day SMAs,
trend direction and maximum drawdown for AAPL, GOOGL, or TSLA.

In [16]:
print("📈 Financial Analysis Query")
print("=" * 50)

response = coordinator.query("How volatile has Apple stock been recently and what is its trend?")
print(response)

📈 Financial Analysis Query


🧭 Routed to ['financial_analysis_tool'] (confidence 1.00)


Financial Analysis:

AAPL [Yahoo Finance (6mo)]:
  Period Return: +30.04%
  Annualized Volatility: 27.21%
  SMA20=334.48 | SMA50=322.28
  Trend: above SMA20, above SMA50
  Max Drawdown: -12.71%


### (c) Portfolio Analysis Tool

`portfolio_analysis_tool` computes cost basis, market value, unrealised P&L, allocation weights,
concentration and risk-tolerance fit for each customer — reporting customer ids only, no PII.

In [17]:
print("💼 Portfolio Analysis Query")
print("=" * 50)

response = coordinator.query(
    "Which customers are sitting on losses and is anyone over-concentrated?"
)
print(response)

💼 Portfolio Analysis Query


🧭 Routed to ['portfolio_analysis_tool'] (confidence 1.00)


Portfolio Analysis:

Customer 1 (moderate, medium risk):
  AAPL: 50.00 sh @ $150.00/sh paid, now $336.67/sh | Total cost $7,500.00 | Value $16,833.50 (Yahoo Finance) | P&L +9,333.50 (+124.45%) | Weight 70.6%
  GOOGL: 20.00 sh @ $2,500.00/sh paid, now $350.50/sh | Total cost $50,000.00 | Value $7,010.00 (Yahoo Finance) | P&L -42,990.00 (-85.98%) | Weight 29.4%
  TOTAL: Cost $57,500.00 | Value $23,843.50 | P&L -33,656.50 (-58.53%)
  ALERT: AAPL 71% of portfolio — over-concentrated for medium risk tolerance (max 60%)

Customer 2 (aggressive, high risk):
  AAPL: 100.00 sh @ $145.00/sh paid, now $336.67/sh | Total cost $14,500.00 | Value $33,667.00 (Yahoo Finance) | P&L +19,167.00 (+132.19%) | Weight 44.3%
  TSLA: 75.00 sh @ $800.00/sh paid, now $377.81/sh | Total cost $60,000.00 | Value $28,335.75 (Yahoo Finance) | P&L -31,664.25 (-52.77%) | Weight 37.3%
  GOOGL: 40.00 sh @ $2,400.00/sh paid, now $350.50/sh | Total cost $96,000.00 | Value $14,020.00 (Yahoo Finance) | P&L -81,980.00 (-85.40

### (d) Confidence Scoring and Observability

After every `coordinator.query()` call, `coordinator.last_trace` holds a dict with the question,
provider used, tools selected, confidence score, a low-confidence flag, the router's reasoning,
latency, token counts and estimated cost.

When confidence falls below `routing.low_confidence_threshold` (0.6) the answer is prefixed
with a one-line notice.

In [18]:
print("🔍 Routing Trace — high-confidence query")
print("=" * 50)
coordinator.query("What is Apple's current stock price?")
import pprint
pprint.pprint(coordinator.last_trace)

print("\n🔍 Routing Trace — ambiguous query (lower confidence)")
print("=" * 50)
response = coordinator.query("Tell me something interesting about the portfolio.")
print("\nAnswer:", response)
print("\nTrace:")
pprint.pprint(coordinator.last_trace)

🔍 Routing Trace — high-confidence query


🧭 Routed to ['finance_market_search_tool'] (confidence 1.00)


{'confidence': 1.0,
 'cost_usd': 0.000675,
 'embedding_tokens': 0,
 'latency_s': 2.47,
 'llm_completion_tokens': 47,
 'llm_prompt_tokens': 1209,
 'provider': 'vocareum',
 'question': "What is Apple's current stock price?",
 'reasoning': "To determine Apple's current stock price, we need to fetch "
              'real-time market data.',
 'tools': ['finance_market_search_tool']}

🔍 Routing Trace — ambiguous query (lower confidence)


🧭 Routed to ['portfolio_analysis_tool', 'financial_analysis_tool', 'database_query_tool'] (confidence 0.70)


   🔒 PII detected ['first_name', 'last_name', 'name'] — applying protection



Answer: The portfolio has a high concentration in AAPL stock across multiple customers, with some exceeding the recommended maximum allocation for their risk tolerance levels. 

Portfolio Analysis:
- Customer 1, 3, 4, 7, and 10 are over-concentrated in AAPL, with weights ranging from 64.1% to 70.6%.
- Customer 2 and 5 have significant exposure to AAPL as well, with weights of 44.3% and 46.2% respectively.
- Customer 6 and 9 are also over-concentrated in AAPL, with weights of 65.8% and 70.6% respectively.

Financial Analysis:
- AAPL has shown a positive period return of +30.04% with an annualized volatility of 27.21%.
- GOOGL has a lower period return of +10.46% and a higher annualized volatility of 34.29%.
- TSLA has a period return of +10.07% and the highest annualized volatility of 49.36%.

Database Results:
- The database query provides the latest close prices for AAPL, GOOGL, and TSLA holdings for each customer.

Trace:
{'confidence': 0.7,
 'cost_usd': 0.003523,
 'embedding_tokens

### (e) Metrics Summary

`coordinator.metrics_summary()` aggregates all traces since startup: query count, average latency,
total tokens, total cost and per-tool usage. Traces are also written to `runtime/metrics.jsonl`.

In [19]:
import pandas as pd

summary = coordinator.metrics_summary()
print("📊 Metrics Summary")
print("=" * 50)

# Scalar metrics as a small DataFrame
scalar_keys = ["queries", "avg_latency_s", "total_tokens", "total_cost_usd", "avg_confidence"]
scalars = pd.DataFrame(
    [(k, summary[k]) for k in scalar_keys if k in summary],
    columns=["metric", "value"]
)
print(scalars.to_string(index=False))

print("\nTool usage:")
tool_df = pd.DataFrame(
    list(summary.get("tool_usage", {}).items()),
    columns=["tool", "calls"]
).sort_values("calls", ascending=False)
print(tool_df.to_string(index=False))

📊 Metrics Summary
        metric       value
       queries    14.00000
 avg_latency_s     5.02000
  total_tokens 35743.00000
total_cost_usd     0.02019
avg_confidence     0.98000

Tool usage:
                      tool  calls
       database_query_tool      6
finance_market_search_tool      4
   portfolio_analysis_tool      4
      AAPL_10k_filing_tool      2
   financial_analysis_tool      2
     GOOGL_10k_filing_tool      1


### (f) Feedback Loop — DSPy Router Optimisation

`coordinator.feedback(question, correct_tools)` records a labelled example.
`coordinator.optimize_router()` runs DSPy BootstrapFewShot over the accumulated examples,
returning before/after accuracy. The compiled router is saved to `runtime/router.json`
and reloaded automatically on startup.

In [20]:
# Record a few labelled examples
coordinator.feedback(
    "How volatile has Tesla been in the last six months?",
    ["financial_analysis_tool"]
)
coordinator.feedback(
    "Which customers have a low-risk investment profile?",
    ["database_query_tool"]
)
coordinator.feedback(
    "What does Google say about AI competition in its 10-K?",
    ["GOOGL_10k_filing_tool"]
)

# Optimise and print before/after accuracy
result = coordinator.optimize_router()
print("🎓 Router optimisation complete")
print(f"  Examples   : {result['examples']}")
print(f"  Accuracy before: {result['accuracy_before']:.0%}")
print(f"  Accuracy after : {result['accuracy_after']:.0%}")

# Re-route one of the feedback questions to confirm the compiled router is live
print("\nRe-routing: 'How volatile has Tesla been in the last six months?'")
coordinator.query("How volatile has Tesla been in the last six months?")
print("  Tools chosen:", coordinator.last_trace["tools"])
print("  Confidence  :", coordinator.last_trace["confidence"])

🎓 Router optimisation complete
  Examples   : 9
  Accuracy before: 100%
  Accuracy after : 100%

Re-routing: 'How volatile has Tesla been in the last six months?'


🧭 Routed to ['financial_analysis_tool'] (confidence 1.00)


  Tools chosen: ['financial_analysis_tool']
  Confidence  : 1.0


## Summary and Key Takeaways

### 🎉 Modular Architecture Achievements

This walkthrough demonstrated the power of the **helper modules architecture**:

#### 🏗️ **Clean Modular Design**
- **DocumentToolsManager**: Focused solely on PDF document analysis
- **FunctionToolsManager**: Handles database, market data, and PII protection
- **AgentCoordinator**: Intelligent orchestration and routing

#### 🧠 **Intelligent Coordination**
- **Automatic Tool Selection**: LLM-based routing to appropriate tools
- **Multi-Tool Synthesis**: Seamless combination of results from different sources
- **Automatic PII Protection**: Smart detection and protection of sensitive data
- **Error Handling**: Graceful degradation and robust error management

#### 🔄 **Flexibility and Reusability**
- **Independent Modules**: Each can be used standalone or together
- **Easy Testing**: Clean interfaces enable comprehensive unit testing
- **Maintainable Code**: Separation of concerns makes updates simple
- **Extensible Design**: Easy to add new tools or capabilities

### 🎯 **Query Types Successfully Demonstrated**

1. **Single Tool Queries**
   - Document analysis (10-K filings)
   - Database queries with automatic SQL generation
   - Real-time market data retrieval

2. **Dual Tool Coordination**
   - Database + Market data combination
   - Document + Database synthesis

3. **Tool Orchestration**
   - Document + Database + Market comprehensive analysis

4. **Automatic PII Protection**
   - Field-based detection and masking
   - Zero performance impact for non-PII queries
   - Transparent coordination between tools

### ⚙️ **Extras (Section 11)**

- **Configuration management**: all model, provider, pricing and routing settings live in `config.toml`;
  `active_provider()` probes Vocareum first and falls back to OpenRouter automatically.
- **Financial analysis**: period return, volatility, SMAs, trend and max drawdown per symbol.
- **Portfolio analysis**: cost basis, market value, unrealised P&L, allocation and risk-fit per customer id.
- **Confidence scoring**: every query trace includes a score; answers below 0.6 get a low-confidence prefix.
- **Observability**: `last_trace` and `metrics_summary()` expose latency, tokens, cost and tool usage;
  traces append to `runtime/metrics.jsonl`.
- **DSPy feedback loop**: `feedback()` + `optimize_router()` improve routing accuracy from labelled examples.